# KrushikaDhara ONNX to TFLite Conversion

Upload `best_model.onnx` and `dataset_split/test/` to the Colab environment before running.

In [ ]:
!pip install onnx onnx-tf tensorflow


In [ ]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf
import numpy as np
import os
from PIL import Image
import glob

onnx_path = "best_model.onnx"
tf_model_path = "saved_model"
tflite_path = "crop_disease_classifier_int8.tflite"
test_dir = "test/" # Path to uploaded test images

print("Loading ONNX model...")
onnx_model = onnx.load(onnx_path)

print("Converting ONNX to TensorFlow SavedModel...")
tf_rep = prepare(onnx_model)
tf_rep.export_graph(tf_model_path)
print(f"SavedModel saved to {tf_model_path}")

print("Converting SavedModel to TFLite (with INT8 quantization)...")
converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)

def representative_dataset():
    image_paths = glob.glob(f"{test_dir}/*/*.*")
    np.random.shuffle(image_paths)
    # Use 100 images for calibration
    for img_path in image_paths[:100]:
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_array = (img_array - mean) / std
            # ONNX-TF expects NCHW since PyTorch used NCHW
            img_array = np.transpose(img_array, (2, 0, 1))
            img_array = np.expand_dims(img_array, axis=0)
            yield [img_array]
        except Exception as e:
            print("Error loading image", e)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"TFLite model saved to {tflite_path}. Download this file.")
